# 03. 파이썬 기초 - 예외처리와 파일 다루기

웹 요청은 **언제든 실패**할 수 있고, 수집한 데이터는 **파일로 저장**해야 합니다.
이 편에서 `try/except`, 파일 입출력, JSON, 환경변수를 익힙니다.

**다루는 내용**
1. 예외처리 try / except / finally
2. 파일 읽기/쓰기 (with open)
3. JSON 다루기 (json 모듈)
4. 경로와 폴더 (os 모듈)
5. 환경변수와 .env (보안)

## 1. 예외처리 try / except

오류가 날 수 있는 코드를 `try` 에 넣고, 오류 발생 시 `except` 로 처리합니다.
프로그램이 멈추지 않게 하는 안전장치입니다.

In [1]:
# 오류가 나면 프로그램이 멈춘다 → try/except 로 감싸 안전하게 처리
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:          # 0으로 나누는 특정 오류만 처리
        print('0으로 나눌 수 없습니다.')
        result = None
    return result

print(safe_divide(10, 2))
print(safe_divide(10, 0))

5.0
0으로 나눌 수 없습니다.
None


In [2]:
# 여러 예외 처리 + finally (성공/실패와 무관하게 항상 실행)
# 실제 스크래핑 코드의 requests 오류 처리와 같은 패턴입니다.
def process(value):
    try:
        number = int(value)          # 숫자로 변환 시도
        print('변환 성공:', number)
    except ValueError as e:          # as e : 오류 객체를 e 에 담아 내용 확인
        print('변환 실패:', e)
    finally:
        print('처리 종료\n')        # 항상 실행 (Selenium 의 driver.quit() 위치)

process('123')
process('abc')

변환 성공: 123
처리 종료

변환 실패: invalid literal for int() with base 10: 'abc'
처리 종료



## 1-1. `raise` — 내가 직접 예외를 일으키기 

지금까지는 파이썬이 **자동으로 발생시킨** 예외를 `try/except` 로 **받았습니다.**
`raise` 는 그 반대로, **내가 직접 예외를 만들어 던지는** 것입니다.

```python
raise ValueError('나이는 숫자여야 합니다')
#     └─ 예외 종류 ─┘ └────── 메시지 ──────┘
```

### 왜 필요한가

잘못된 값을 `None` 으로 돌려주고 넘어가면, 오류가 **한참 뒤 엉뚱한 곳에서** 터집니다.
`raise` 는 **문제를 발견한 그 자리에서 즉시** 알려 원인 추적을 쉽게 만듭니다.

> **핵심: `raise` 는 "에러를 내는 것" 이 아니라 "호출한 쪽에 문제를 알리는 것" 입니다.**
> 함수가 약속(계약)을 지킬 수 없을 때 조용히 넘어가지 않고 명확히 신고하는 장치입니다.

### 자주 쓰는 내장 예외

| 예외 | 언제 던지나 | 예 |
|---|---|---|
| `ValueError` | 타입은 맞는데 **값**이 잘못됨 | 나이가 `-5`, 빈 검색어 |
| `TypeError` | **타입** 자체가 잘못됨 | 숫자 자리에 문자열 |
| `KeyError` | 딕셔너리에 **키가 없음** | 없는 사용자 id |
| `FileNotFoundError` | 파일이 없음 | 설정 파일 누락 |
| `RuntimeError` | 위에 해당 없는 실행 중 오류 | 설정 로딩 실패 |

In [1]:
# =========================================================
# raise - 내가 직접 예외를 일으키기
#   목적: 잘못된 값을 '발견한 그 자리에서 즉시' 알리는 것
# =========================================================

# 방법 1) 잘못되면 None 을 돌려주기 → 문제를 뒤로 미룬다
def get_age_bad(value):
    """나이를 정수로 변환한다. 실패하면 None 을 돌려준다."""
    try:
        return int(value)
    except ValueError:
        return None


# 방법 2) raise 로 즉시 알리기 → 문제가 생긴 자리에서 드러난다
def get_age(value):
    """나이 문자열을 정수로 변환한다.

    Args:
        value (str): 나이 문자열

    Returns:
        int: 0~150 사이의 나이

    Raises:
        ValueError: 숫자가 아니거나 범위를 벗어난 경우
    """
    try:
        age = int(value)
    except ValueError:
        # raise 뒤에 '예외 객체' 를 만들어 던진다. 메시지는 원인이 보이게 쓴다
        raise ValueError(f'나이는 숫자여야 합니다: {value!r}')

    # 형식은 맞지만 값이 이상한 경우도 직접 걸러낸다
    if not 0 <= age <= 150:
        raise ValueError(f'나이 범위를 벗어났습니다: {age}')
    return age


print('정상 동작:', get_age('25'))

print()
# None 을 돌려주면 오류가 '한참 뒤 엉뚱한 곳' 에서 터진다
bad = get_age_bad('스물다섯')
print('[None 반환 방식] 반환값:', bad)
try:
    print('  내년 나이 계산:', bad + 1)
except TypeError as exp:
    print('  → 뒤늦게 TypeError:', exp)
    print('  → 진짜 원인(잘못된 입력)이 어디였는지 알 수 없다')

print()
# raise 방식은 원인과 위치를 그 자리에서 알려준다
for v in ['스물다섯', '200']:
    try:
        get_age(v)
    except ValueError as exp:
        print(f'[raise 방식] {v!r} → ValueError: {exp}')

정상 동작: 25

[None 반환 방식] 반환값: None
  → 뒤늦게 TypeError: unsupported operand type(s) for +: 'NoneType' and 'int'
  → 진짜 원인(잘못된 입력)이 어디였는지 알 수 없다

[raise 방식] '스물다섯' → ValueError: 나이는 숫자여야 합니다: '스물다섯'
[raise 방식] '200' → ValueError: 나이 범위를 벗어났습니다: 200


In [4]:
# =========================================================
# 상황에 맞는 '예외 종류' 고르기
#   메시지만큼 어떤 종류의 예외를 던지느냐도 중요하다.
#   호출하는 쪽이 except 로 골라 잡아 다르게 대응할 수 있기 때문이다.
# =========================================================
def find_user(users, user_id):
    """사용자를 찾아 이름을 돌려준다.

    Args:
        users (dict): {id: 이름} 형태의 사용자 목록
        user_id (int): 찾을 사용자 id

    Returns:
        str: 사용자 이름

    Raises:
        TypeError: user_id 가 정수가 아닌 경우 (타입 자체가 잘못됨)
        KeyError: 해당 id 의 사용자가 없는 경우 (키가 없음)
    """
    if not isinstance(user_id, int):
        raise TypeError(f'user_id 는 정수여야 합니다 (받은 타입: {type(user_id).__name__})')
    if user_id not in users:
        raise KeyError(f'존재하지 않는 사용자 id: {user_id}')
    return users[user_id]


users = {1: '홍길동', 2: '김철수'}

# 호출하는 쪽은 '예외 종류' 로 구분해 각각 다르게 대응한다
for arg in [1, '1', 99]:
    try:
        print(f'  {arg!r:>5} → {find_user(users, arg)}')
    except TypeError as exp:
        print(f'  {arg!r:>5} → [입력 형식 오류] {exp}')     # 코드를 고쳐야 하는 문제
    except KeyError as exp:
        print(f'  {arg!r:>5} → [데이터 없음] {exp}')        # 데이터 문제

# 참고: KeyError 는 메시지를 출력할 때 따옴표가 함께 붙는다 (KeyError 의 특성)

      1 → 홍길동
    '1' → [입력 형식 오류] user_id 는 정수여야 합니다 (받은 타입: str)
     99 → [데이터 없음] '존재하지 않는 사용자 id: 99'


### 예외를 다시 던지거나, 바꿔서 던지기

| 문법 | 하는 일 | 언제 |
|---|---|---|
| `raise` (인자 없이) | 방금 잡은 예외를 **그대로** 다시 던짐 | 로그만 남기고 처리는 호출한 쪽에 맡길 때 |
| `raise 새예외 from 원본` | **다른 예외로 바꾸되 원인을 연결** | 저수준 예외를 우리 용어로 감쌀 때 |

`from` 을 쓰면 원본 예외가 `__cause__` 에 보존되고,
traceback 에 *"The above exception was the direct cause of the following exception"* 으로 함께 표시됩니다.

> `except` 안에서 `raise` 를 쓸 때 **`raise exp` 라고 쓰지 마세요.**
> 인자 없이 `raise` 만 쓰면 원래 traceback 이 그대로 보존되지만,
> `raise exp` 는 그 위치에서 다시 시작된 것처럼 기록되어 추적이 어려워집니다.

In [2]:
# =========================================================
# 1) raise 단독 - 잡았던 예외를 '그대로' 다시 던지기
#    로그만 남기고, 실제 처리는 호출한 쪽에 맡길 때 쓴다.
# =========================================================
def load_config(path):
    """설정 파일을 읽는다. 없으면 로그를 남기고 예외를 다시 던진다.

    Raises:
        FileNotFoundError: 파일이 없는 경우 (그대로 다시 던짐)
    """
    try:
        with open(path, encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        print(f'  [로그] 설정 파일 없음: {path}')
        raise      # 인자 없이 raise → 방금 잡은 예외를 그대로 다시 던진다


print('1) raise 단독')
try:
    load_config('없는파일.txt')
except FileNotFoundError as exp:
    print('  호출한 쪽에서 처리:', exp.strerror)



1) raise 단독
  [로그] 설정 파일 없음: 없는파일.txt
  호출한 쪽에서 처리: No such file or directory


In [ ]:

# =========================================================
# 2) raise ... from exp - 원인을 연결해 '다른 예외로 바꿔' 던지기
#    저수준 예외(FileNotFoundError)를 우리 프로젝트 용어의 예외로 감쌀 때 쓴다.
# =========================================================
def load_settings(path):
    """설정을 읽되, 실패하면 RuntimeError 로 바꿔 던진다.

    Raises:
        RuntimeError: 설정을 불러오지 못한 경우 (원인은 __cause__ 에 보존)
    """
    try:
        with open(path, encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError as exp:
        # from exp : 원래 원인을 잃지 않고 연결해 둔다
        raise RuntimeError('설정을 불러오지 못했습니다') from exp


print('2) raise ... from')
try:
    load_settings('없는파일.txt')
except RuntimeError as exp:
    print('  겉으로 드러난 예외 :', type(exp).__name__, '-', exp)
    print('  실제 원인(__cause__):', type(exp.__cause__).__name__, '-', exp.__cause__)

print()
print('→ from 을 생략하면 원인이 끊겨 "왜 실패했는지" 를 추적할 수 없다')

2) raise ... from
  겉으로 드러난 예외 : RuntimeError - 설정을 불러오지 못했습니다
  실제 원인(__cause__): FileNotFoundError - [Errno 2] No such file or directory: '없는파일.txt'

→ from 을 생략하면 원인이 끊겨 "왜 실패했는지" 를 추적할 수 없다


In [6]:
# =========================================================
# 사용자 정의 예외 - 내 프로젝트만의 오류 종류 만들기
#   Exception 을 상속하면 끝이다. 본문은 docstring 하나면 충분하다.
#   장점: 이름 자체가 문서가 되고, 호출하는 쪽이 정확히 골라 잡을 수 있다.
# =========================================================

class ScrapingError(Exception):
    """스크래핑 과정에서 발생하는 오류의 최상위 클래스."""


class PageNotFoundError(ScrapingError):
    """요청한 페이지가 없을 때."""


class ParseError(ScrapingError):
    """HTML 구조가 예상과 달라 파싱에 실패했을 때."""


def scrape(status_code, html):
    """응답을 검사하고 <title> 내용을 파싱한다.

    Args:
        status_code (int): HTTP 응답 코드
        html (str): 응답 본문

    Returns:
        str: title 태그 안의 텍스트

    Raises:
        PageNotFoundError: 404 응답인 경우
        ParseError: title 태그를 찾지 못한 경우
    """
    if status_code == 404:
        raise PageNotFoundError(f'페이지 없음 (status={status_code})')
    if '<title>' not in html:
        raise ParseError('title 태그를 찾을 수 없습니다')
    return html.split('<title>')[1].split('</title>')[0]


cases = [
    (200, '<html><title>파이썬</title></html>'),   # 정상
    (404, ''),                                     # 페이지 없음
    (200, '<html>제목 없음</html>'),                # 구조가 다름
]

print('예외 종류별로 다르게 대응하기:')
for code, html in cases:
    try:
        print('  성공:', scrape(code, html))
    except PageNotFoundError as exp:
        print('  건너뜀:', exp)          # 이 페이지만 넘어가고 계속 진행
    except ParseError as exp:
        print('  파싱 실패:', exp)        # 셀렉터 점검이 필요한 상황

print()
# 상속 관계 덕분에 상위 클래스 하나로 전부 잡을 수도 있다
print('ScrapingError 하나로 모두 잡기:')
for code, html in cases:
    try:
        scrape(code, html)
    except ScrapingError as exp:
        print(f'  {type(exp).__name__}: {exp}')

예외 종류별로 다르게 대응하기:
  성공: 파이썬
  건너뜀: 페이지 없음 (status=404)
  파싱 실패: title 태그를 찾을 수 없습니다

ScrapingError 하나로 모두 잡기:
  PageNotFoundError: 페이지 없음 (status=404)
  ParseError: title 태그를 찾을 수 없습니다


## 2. 파일 읽기/쓰기 — with open

`with open(...)` 을 쓰면 파일을 자동으로 닫아줘 안전합니다.
한글이 깨지지 않도록 `encoding='utf-8'` 을 지정합니다.

In [7]:
# 파일 쓰기 ('w' = write, 기존 내용 덮어씀)
with open('sample.txt', 'w', encoding='utf-8') as f:
    f.write('첫번째 줄\n')
    f.write('두번째 줄\n')
print('파일 저장 완료')

# 파일 읽기 ('r' = read)
with open('sample.txt', 'r', encoding='utf-8') as f:
    content = f.read()
print(content)

파일 저장 완료
첫번째 줄
두번째 줄



## 3. JSON 다루기

API 응답과 `data/` 폴더의 `.json` 파일은 JSON 형식입니다.
- `json.dump` : 파이썬 객체 → JSON 파일로 저장
- `json.load` : JSON 파일 → 파이썬 객체로 읽기

In [8]:
import json

# 저장할 데이터 (딕셔너리들의 리스트 — 01편에서 배운 구조)
songs = [
    {'rank': 1, 'title': 'Dynamite', 'artist': 'BTS'},
    {'rank': 2, 'title': 'How You Like That', 'artist': 'BLACKPINK'},
]

# JSON 파일로 저장
# ensure_ascii=False : 한글을 그대로 저장 (True면 \uXXXX 로 깨져 보임)
# indent=2 : 사람이 읽기 좋게 들여쓰기
with open('songs_sample.json', 'w', encoding='utf-8') as f:
    json.dump(songs, f, ensure_ascii=False, indent=2)
print('JSON 저장 완료')

JSON 저장 완료


In [9]:
# JSON 파일 읽기 → 다시 파이썬 리스트/딕셔너리로
with open('songs_sample.json', 'r', encoding='utf-8') as f:
    loaded = json.load(f)

print(type(loaded))          # list
print(loaded[0]['title'])    # 첫 곡 제목
for s in loaded:
    print(s['rank'], s['title'], '-', s['artist'])

<class 'list'>
Dynamite
1 Dynamite - BTS
2 How You Like That - BLACKPINK


## 4. 경로와 폴더 — os 모듈

파일을 저장하기 전에 폴더가 있는지 확인하고 없으면 만듭니다.
스크래핑 코드의 `os.makedirs('data', exist_ok=True)` 가 그 예입니다.

In [10]:
import os

# 폴더가 없으면 만들기 (exist_ok=True : 이미 있어도 오류 안 남)
os.makedirs('output_sample', exist_ok=True)
print('폴더 준비 완료')

# 경로 합치기 (운영체제에 맞게 / 또는 \ 처리)
path = os.path.join('output_sample', 'result.txt')
print('경로:', path)

# 파일/폴더 존재 확인
print('sample.txt 존재?', os.path.exists('sample.txt'))

폴더 준비 완료
경로: output_sample\result.txt
sample.txt 존재? True


## 5. 환경변수와 .env — 비밀정보 다루기

API 키·비밀번호를 코드에 직접 쓰면 위험합니다.
`.env` 파일에 넣고 `os.getenv()` 로 불러오면 코드와 비밀정보를 분리할 수 있습니다.
(이 프로젝트의 `streamlit_book_search.py` 가 이 방식을 사용합니다.)

In [11]:
# .env 파일 예시 (실제로는 아래 내용을 프로젝트 루트의 .env 에 저장)
#   CLIENT_ID=여기에_아이디
#   CLIENT_SECRET=여기에_비밀키

# python-dotenv 라이브러리로 .env 를 읽어 환경변수로 등록
from dotenv import load_dotenv
import os

load_dotenv()   # .env 파일을 읽어옴

# os.getenv('키') 로 값을 꺼낸다 (없으면 None)
client_id = os.getenv('CLIENT_ID')

# 비밀정보는 전체를 출력하지 말고 일부만 확인
if client_id:
    print('CLIENT_ID 앞 4자리:', client_id[:4])
else:
    print('CLIENT_ID 가 설정되어 있지 않습니다. (.env 확인)')

# ⚠️ 주의: .env 파일은 .gitignore 에 넣어 Git 에 올리지 않아야 합니다.

CLIENT_ID 앞 4자리: ic0k


## 정리
- **try/except/finally**: 오류에 안전한 코드 (requests 실패 대비)
- **raise**: 잘못된 값을 **발견한 자리에서 즉시** 알리기
- **with open(..., encoding='utf-8')**: 파일 자동 닫힘 + 한글 처리
- **json.dump / json.load**: 파이썬 객체 ↔ JSON 파일
- **os.makedirs / path.join / path.exists**: 폴더·경로 안전 처리
- **dotenv + os.getenv**: 비밀정보를 코드와 분리

### `raise` 요약

| 문법 | 하는 일 |
|---|---|
| `raise ValueError('메시지')` | 새 예외를 만들어 던진다 |
| `raise` (인자 없이) | 방금 잡은 예외를 **그대로** 다시 던진다 (traceback 보존) |
| `raise 새예외 from 원본` | 다른 예외로 바꾸되 **원인을 `__cause__` 에 연결** |
| `class MyError(Exception)` | 프로젝트 전용 예외 정의 (상속으로 묶어 관리) |

**기억할 것**
- `try/except` 는 예외를 **받는 쪽**, `raise` 는 예외를 **보내는 쪽**
- `None` 을 돌려주고 넘어가면 오류가 **뒤늦게 엉뚱한 곳**에서 터진다
- 메시지에 **어떤 값이 문제였는지**(`{value!r}`)를 넣으면 디버깅이 쉬워진다

다음: `04python_basic_pandas기초.ipynb` (표 데이터 분석)